# OSINT Entity Resolution - Data Exploration

This notebook explores the entity resolution dataset to understand:
- Data structure and schema
- Class balance (positive vs negative matches)
- Entity attributes and completeness
- Data quality issues

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Add scripts to path
sys.path.append('..')
from scripts.load_data import load_pairs, create_sample, load_sample

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Create Sample Data

First, let's create a small sample to explore quickly.

In [ ]:
# Create sample if it doesn't exist
sample_path = Path('../data/sample_1000.json')
if not sample_path.exists():
    create_sample('../pairs-20251209.json.gz', str(sample_path), n=1000)

# Load sample
sample_data = load_sample(sample_path)
print(f"Loaded {len(sample_data)} samples")

## 2. Inspect Data Structure

Let's look at a few examples to understand the schema.

In [ ]:
# Display first example (positive match)
print("=" * 80)
print("EXAMPLE 1: Full structure")
print("=" * 80)
print(json.dumps(sample_data[0], indent=2))

In [ ]:
# Inspect the keys in entity data
print("Keys in 'left' entity:")
print(list(sample_data[0]['left'].keys()))
print("\nKeys in 'right' entity:")
print(list(sample_data[0]['right'].keys()))

## 3. Class Balance

Examine the distribution of positive vs negative matches.

In [ ]:
# Count judgements
judgements = [pair['judgement'] for pair in sample_data]
judgement_counts = Counter(judgements)

print("Class Distribution:")
for label, count in judgement_counts.items():
    pct = (count / len(sample_data)) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

# Visualize
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
ax[0].bar(judgement_counts.keys(), judgement_counts.values())
ax[0].set_xlabel('Judgement')
ax[0].set_ylabel('Count')
ax[0].set_title('Class Distribution')

# Pie chart
ax[1].pie(judgement_counts.values(), labels=judgement_counts.keys(), autopct='%1.1f%%')
ax[1].set_title('Class Balance')

plt.tight_layout()
plt.show()

## 4. Entity Attribute Analysis

Analyze which attributes are present and how complete they are.

In [ ]:
# Collect all unique keys from left and right entities
all_left_keys = set()
all_right_keys = set()

for pair in sample_data:
    all_left_keys.update(pair['left'].keys())
    all_right_keys.update(pair['right'].keys())

print(f"Unique attributes in 'left' entities: {len(all_left_keys)}")
print(f"Unique attributes in 'right' entities: {len(all_right_keys)}")
print(f"\nCommon attributes: {all_left_keys & all_right_keys}")
print(f"Only in left: {all_left_keys - all_right_keys}")
print(f"Only in right: {all_right_keys - all_left_keys}")

In [ ]:
# Calculate attribute completeness (non-null, non-empty values)
def calculate_completeness(entities, attribute):
    """Calculate % of entities that have a non-empty value for attribute."""
    count = sum(
        1 for entity in entities 
        if attribute in entity and entity[attribute] and str(entity[attribute]).strip()
    )
    return (count / len(entities)) * 100 if entities else 0

# Get all entities
left_entities = [pair['left'] for pair in sample_data]
right_entities = [pair['right'] for pair in sample_data]

# Calculate completeness for common attributes
common_attrs = sorted(all_left_keys & all_right_keys)
completeness_data = []

for attr in common_attrs:
    left_comp = calculate_completeness(left_entities, attr)
    right_comp = calculate_completeness(right_entities, attr)
    completeness_data.append({
        'attribute': attr,
        'left_completeness': left_comp,
        'right_completeness': right_comp,
        'avg_completeness': (left_comp + right_comp) / 2
    })

# Create DataFrame and display
df_completeness = pd.DataFrame(completeness_data)
df_completeness = df_completeness.sort_values('avg_completeness', ascending=False)
print("\nAttribute Completeness (%):\n")
print(df_completeness.to_string(index=False))

In [ ]:
# Visualize top attributes by completeness
top_n = 10
df_top = df_completeness.head(top_n)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df_top))
width = 0.35

ax.barh(x - width/2, df_top['left_completeness'], width, label='Left Entity')
ax.barh(x + width/2, df_top['right_completeness'], width, label='Right Entity')

ax.set_ylabel('Attribute')
ax.set_xlabel('Completeness (%)')
ax.set_title(f'Top {top_n} Most Complete Attributes')
ax.set_yticks(x)
ax.set_yticklabels(df_top['attribute'])
ax.legend()
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()

## 5. Positive vs Negative Pair Comparison

Look at differences between positive and negative matches.

In [ ]:
# Separate positive and negative pairs
positive_pairs = [pair for pair in sample_data if pair['judgement'] == 'positive']
negative_pairs = [pair for pair in sample_data if pair['judgement'] == 'negative']

print(f"Positive pairs: {len(positive_pairs)}")
print(f"Negative pairs: {len(negative_pairs)}")

# Display example of each
if positive_pairs:
    print("\n" + "="*80)
    print("EXAMPLE POSITIVE MATCH:")
    print("="*80)
    print(json.dumps(positive_pairs[0], indent=2)[:500] + "...")

if negative_pairs:
    print("\n" + "="*80)
    print("EXAMPLE NEGATIVE MATCH:")
    print("="*80)
    print(json.dumps(negative_pairs[0], indent=2)[:500] + "...")

## 6. Next Steps

Based on this EDA, consider:

1. **Data Preprocessing**: Which attributes are most useful for entity resolution?
2. **Feature Engineering**: How to represent entity pairs for ML models?
3. **Baseline Approach**: What's a simple LLM prompt structure?
4. **Evaluation Strategy**: How to split data? Stratified sampling?

---

**Add your own analysis below:**

In [ ]:
# Your custom analysis here
